# Step 10: BERT — Bidirectional Encoder from Transformers

GPT learns to predict the next token — so its attention is strictly left-to-right.
Many tasks don't need to generate text; they need to **understand** it.
For understanding, seeing the full sentence before deciding anything is better.

**BERT** (Devlin et al., 2018) flips the pre-training objective:
- **Masked Language Modeling (MLM)**: mask ~15% of tokens, predict them from both directions
- **Next Sentence Prediction (NSP)**: predict whether two sentences are consecutive

The result: a bidirectional encoder that learns rich contextual representations.
Fine-tuning with a task-specific head achieves SOTA on 11 NLP benchmarks simultaneously.

**Architecture**: identical to GPT encoder blocks — but **no causal mask** (every position
attends to every other), plus a `[CLS]` token at the start and `[SEP]` between sentences.

**What you'll see:**
1. MLM pre-training: masking strategy and prediction objective
2. A full BERT-style encoder
3. Pre-training from scratch on a small corpus
4. Fine-tuning the pre-trained encoder on a classification task
5. Pre-trained vs. from-scratch comparison: the value of pre-training

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import matplotlib.pyplot as plt

torch.manual_seed(42)
np.random.seed(42)
plt.rcParams['figure.dpi'] = 100

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## The Masking Strategy

For each 15%-selected token:
- 80% of the time: replace with `[MASK]`
- 10% of the time: replace with a random token
- 10% of the time: keep the original token

The randomization and keep-original strategies prevent the model from learning
"if it's `[MASK]`, predict something" — instead it learns to use context for **all** tokens.
This means the representations learned are useful for all positions, not just masked ones.

In [ ]:
# Special token IDs
PAD_ID  = 0
CLS_ID  = 1
SEP_ID  = 2
MASK_ID = 3
N_SPECIAL = 4  # number of special tokens before vocabulary

def mask_tokens(input_ids, vocab_size, mask_prob=0.15):
    """
    Apply BERT masking strategy.
    Returns: masked_ids, labels (original id where masked, -100 elsewhere)
    """
    labels = input_ids.clone()
    masked = input_ids.clone()

    # Create mask for which tokens are eligible (not special tokens)
    special = (input_ids == CLS_ID) | (input_ids == SEP_ID) | (input_ids == PAD_ID)
    # Randomly select 15% of non-special tokens
    prob_matrix = torch.full(input_ids.shape, mask_prob)
    prob_matrix[special] = 0.0
    selected = torch.bernoulli(prob_matrix).bool()

    # -100 tells CrossEntropyLoss to ignore non-masked positions
    labels[~selected] = -100

    # 80% → [MASK]
    replace_with_mask   = selected & (torch.rand(input_ids.shape) < 0.8)
    masked[replace_with_mask] = MASK_ID

    # 10% → random token (remaining 20% × 50%)
    replace_with_random = selected & ~replace_with_mask & (torch.rand(input_ids.shape) < 0.5)
    rand_ids = torch.randint(N_SPECIAL, vocab_size, input_ids.shape)
    masked[replace_with_random] = rand_ids[replace_with_random]

    # Remaining 10%: keep original (already in masked)
    return masked, labels


# Demo
demo_input = torch.tensor([[CLS_ID, 10, 11, 12, 13, 14, 15, 16, SEP_ID]])
demo_masked, demo_labels = mask_tokens(demo_input, vocab_size=50)
print("Original: ", demo_input[0].tolist())
print("Masked:   ", demo_masked[0].tolist(), "  (3 = [MASK])")
print("Labels:   ", demo_labels[0].tolist(), "  (-100 = ignored)")

## The BERT Architecture

BERT is a Transformer **encoder** — every position attends to every other position
(no causal mask). The architecture is identical to GPT's blocks except:
1. No causal mask
2. Segment embeddings (sentence A vs. B)
3. `[CLS]` and `[SEP]` as input tokens

The `[CLS]` token's final representation is used as the **sentence-level embedding**
for classification tasks — the model learns to aggregate information from the whole
sequence into this single vector during fine-tuning.

In [ ]:
class MultiHeadSelfAttention(nn.Module):
    """Full bidirectional self-attention (no causal mask)."""
    def __init__(self, d_model, n_heads, dropout=0.1):
        super().__init__()
        assert d_model % n_heads == 0
        self.d_k = d_model // n_heads
        self.n_heads = n_heads
        self.Wqkv = nn.Linear(d_model, 3 * d_model, bias=False)
        self.Wo   = nn.Linear(d_model, d_model, bias=False)
        self.drop = nn.Dropout(dropout)

    def forward(self, x, key_padding_mask=None):
        B, T, D = x.shape
        H, dk = self.n_heads, self.d_k

        Q, K, V = self.Wqkv(x).split(D, dim=-1)
        Q = Q.view(B, T, H, dk).transpose(1, 2)
        K = K.view(B, T, H, dk).transpose(1, 2)
        V = V.view(B, T, H, dk).transpose(1, 2)

        scores = Q @ K.transpose(-2, -1) / math.sqrt(dk)  # (B, H, T, T)
        if key_padding_mask is not None:
            # key_padding_mask: (B, T) — True for padding positions
            scores = scores.masked_fill(
                key_padding_mask.unsqueeze(1).unsqueeze(2), float('-inf'))
        weights = self.drop(F.softmax(scores, dim=-1))
        out = (weights @ V).transpose(1, 2).contiguous().view(B, T, D)
        return self.Wo(out)


class BERTBlock(nn.Module):
    """Pre-LN bidirectional Transformer block."""
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.ln1  = nn.LayerNorm(d_model)
        self.attn = MultiHeadSelfAttention(d_model, n_heads, dropout)
        self.ln2  = nn.LayerNorm(d_model)
        self.ff   = nn.Sequential(
            nn.Linear(d_model, d_ff), nn.GELU(), nn.Linear(d_ff, d_model),
            nn.Dropout(dropout)
        )

    def forward(self, x, key_padding_mask=None):
        x = x + self.attn(self.ln1(x), key_padding_mask)
        x = x + self.ff(self.ln2(x))
        return x


class BERT(nn.Module):
    def __init__(self, vocab_size, d_model, n_heads, d_ff, n_layers,
                 max_len=128, dropout=0.1):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size, d_model, padding_idx=PAD_ID)
        self.pos_emb = nn.Embedding(max_len, d_model)
        self.drop    = nn.Dropout(dropout)
        self.blocks  = nn.ModuleList([
            BERTBlock(d_model, n_heads, d_ff, dropout)
            for _ in range(n_layers)
        ])
        self.ln_f    = nn.LayerNorm(d_model)
        # MLM head: predict the masked token
        self.mlm_head = nn.Linear(d_model, vocab_size, bias=False)
        self.mlm_head.weight = self.tok_emb.weight  # weight tying

        self.apply(self._init)

    def _init(self, m):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, std=0.02)
            if m.bias is not None: nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, std=0.02)

    def encode(self, x):
        """Return all hidden states — shared by both pre-training and fine-tuning."""
        B, T = x.shape
        pad_mask = (x == PAD_ID)  # (B, T) — True where padding
        pos = torch.arange(T, device=x.device).unsqueeze(0)
        h = self.drop(self.tok_emb(x) + self.pos_emb(pos))
        for block in self.blocks:
            h = block(h, pad_mask)
        return self.ln_f(h)  # (B, T, d_model)

    def forward_mlm(self, x):
        """Pre-training forward: predict all token positions."""
        h = self.encode(x)           # (B, T, d_model)
        return self.mlm_head(h)      # (B, T, vocab_size)

    def cls_embedding(self, x):
        """Fine-tuning forward: return [CLS] embedding."""
        h = self.encode(x)           # (B, T, d_model)
        return h[:, 0]               # (B, d_model) — [CLS] position


VOCAB_SIZE = 50
D_MODEL    = 64
N_HEADS    = 4
D_FF       = 256
N_LAYERS   = 2
MAX_LEN    = 32

bert = BERT(VOCAB_SIZE, D_MODEL, N_HEADS, D_FF, N_LAYERS, MAX_LEN).to(device)
total_params = sum(p.numel() for p in bert.parameters())
print(f"BERT parameters: {total_params:,}")

# Verify shapes
x = torch.randint(N_SPECIAL, VOCAB_SIZE, (2, 12)).to(device)
x[:, 0] = CLS_ID; x[:, -1] = SEP_ID
logits = bert.forward_mlm(x)
cls_emb = bert.cls_embedding(x)
print(f"MLM output:  {logits.shape}  (B, T, vocab_size)")
print(f"CLS output:  {cls_emb.shape}  (B, d_model)")

In [ ]:
# Pre-training corpus: same Shakespeare excerpt for consistency
raw_text = """To be or not to be that is the question
Whether tis nobler in the mind to suffer
The slings and arrows of outrageous fortune
Or to take arms against a sea of troubles
And by opposing end them to die to sleep
No more and by a sleep to say we end
The heartache and the thousand natural shocks
That flesh is heir to tis a consummation
Devoutly to be wished to die to sleep
To sleep perchance to dream aye theres the rub
For in that sleep of death what dreams may come
When we have shuffled off this mortal coil
Must give us pause there is the respect
That makes calamity of so long life"""

# Character-level tokenizer with special tokens
chars = sorted(set(raw_text))
char2id = {c: i + N_SPECIAL for i, c in enumerate(chars)}
id2char = {v: k for k, v in char2id.items()}
id2char.update({CLS_ID: '[CLS]', SEP_ID: '[SEP]', MASK_ID: '[MASK]', PAD_ID: '[PAD]'})

VOCAB_SIZE = len(chars) + N_SPECIAL
print(f"Vocabulary: {VOCAB_SIZE} tokens ({len(chars)} chars + {N_SPECIAL} special)")

# Rebuild BERT with correct vocab size
bert = BERT(VOCAB_SIZE, D_MODEL, N_HEADS, D_FF, N_LAYERS, MAX_LEN).to(device)

# Encode raw text into fixed-length chunks with [CLS] and [SEP]
all_ids = [char2id[c] for c in raw_text]
CHUNK = MAX_LEN - 2  # reserve 2 positions for CLS and SEP

chunks = []
for i in range(0, len(all_ids) - CHUNK, CHUNK // 2):  # 50% overlap
    chunk = [CLS_ID] + all_ids[i:i+CHUNK] + [SEP_ID]
    if len(chunk) < MAX_LEN:
        chunk += [PAD_ID] * (MAX_LEN - len(chunk))
    chunks.append(chunk[:MAX_LEN])

corpus = torch.tensor(chunks, dtype=torch.long)
print(f"Corpus chunks: {len(corpus)} sequences of length {MAX_LEN}")

In [ ]:
# Pre-training loop: MLM objective
optimizer_pretrain = torch.optim.AdamW(bert.parameters(), lr=3e-4, weight_decay=0.01)
pretrain_losses = []
N_PRETRAIN = 1500
BATCH_SIZE = 8

print("Pre-training BERT on MLM...")
for step in range(N_PRETRAIN):
    # Random batch
    idx = torch.randint(0, len(corpus), (BATCH_SIZE,))
    batch = corpus[idx].to(device)

    masked_batch, labels = mask_tokens(batch, VOCAB_SIZE)
    masked_batch = masked_batch.to(device)
    labels = labels.to(device)

    logits = bert.forward_mlm(masked_batch)        # (B, T, vocab)
    # Only compute loss on masked positions (labels == -100 are ignored)
    loss = F.cross_entropy(logits.view(-1, VOCAB_SIZE), labels.view(-1),
                           ignore_index=-100)

    optimizer_pretrain.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(bert.parameters(), 1.0)
    optimizer_pretrain.step()
    pretrain_losses.append(loss.item())

    if (step + 1) % 300 == 0:
        print(f"  Step {step+1} | MLM loss: {loss.item():.4f}")

print("Pre-training complete.")

# Save a copy of pre-trained weights
pretrained_state = {k: v.clone() for k, v in bert.state_dict().items()}

## Fine-Tuning: Sequence Classification

After pre-training, we add a small classification head on top of the `[CLS]` embedding
and fine-tune on labeled examples. The Transformer weights are updated jointly.

**Task**: given a sequence, classify whether it starts with a capital letter.
This is a proxy for real downstream tasks — simple enough to train on tiny data.

We compare:
1. **Fine-tune from pre-trained BERT**: start with learned representations
2. **Train from scratch**: same architecture, random initialization, same fine-tuning data

In [ ]:
class BERTForClassification(nn.Module):
    def __init__(self, bert_encoder, d_model, n_classes, dropout=0.1):
        super().__init__()
        self.bert = bert_encoder
        self.head = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(d_model, n_classes)
        )

    def forward(self, x):
        cls = self.bert.cls_embedding(x)  # (B, d_model)
        return self.head(cls)             # (B, n_classes)


def make_classification_data(n=200, seq_len=MAX_LEN):
    """Binary task: does the sequence contain more chars from the first half of alphabet?"""
    all_char_ids = list(char2id.values())
    low_ids  = [cid for c, cid in char2id.items() if 'a' <= c <= 'm']
    high_ids = [cid for c, cid in char2id.items() if 'n' <= c <= 'z' or 'A' <= c <= 'Z']

    seqs, labels = [], []
    for _ in range(n):
        # Random character sequence
        length = np.random.randint(6, CHUNK)
        token_ids = np.random.choice(all_char_ids, size=length).tolist()
        # Label: 1 if sequence has more 'high' chars, 0 otherwise
        low_count  = sum(1 for t in token_ids if t in low_ids)
        high_count = sum(1 for t in token_ids if t in high_ids)
        label = 1 if high_count > low_count else 0

        seq = [CLS_ID] + token_ids[:CHUNK] + [SEP_ID]
        seq += [PAD_ID] * (MAX_LEN - len(seq))
        seqs.append(seq[:MAX_LEN])
        labels.append(label)

    return torch.tensor(seqs, dtype=torch.long), torch.tensor(labels, dtype=torch.long)


def finetune(bert_model, n_steps=400, lr=1e-4):
    """Fine-tune a BERTForClassification on the proxy task."""
    clf = BERTForClassification(bert_model, D_MODEL, n_classes=2, dropout=0.1).to(device)
    opt = torch.optim.AdamW(clf.parameters(), lr=lr, weight_decay=0.01)

    x_train, y_train = make_classification_data(n=200)
    x_val,   y_val   = make_classification_data(n=100)
    x_train, y_train = x_train.to(device), y_train.to(device)
    x_val,   y_val   = x_val.to(device),   y_val.to(device)

    train_losses, val_accs = [], []
    for step in range(n_steps):
        idx = torch.randint(0, len(x_train), (16,))
        logits = clf(x_train[idx])
        loss = F.cross_entropy(logits, y_train[idx])
        opt.zero_grad(); loss.backward(); opt.step()
        train_losses.append(loss.item())

        if (step + 1) % 50 == 0:
            clf.eval()
            with torch.no_grad():
                val_logits = clf(x_val)
                acc = (val_logits.argmax(-1) == y_val).float().mean().item()
            val_accs.append((step, acc))
            clf.train()

    return train_losses, val_accs


# Fine-tune from pre-trained BERT
print("Fine-tuning from pre-trained BERT...")
ft_losses, ft_accs = finetune(bert, n_steps=400)

# Train from scratch: reset weights, fine-tune on the same labeled data
bert_scratch = BERT(VOCAB_SIZE, D_MODEL, N_HEADS, D_FF, N_LAYERS, MAX_LEN).to(device)
print("Training from scratch (no pre-training)...")
scratch_losses, scratch_accs = finetune(bert_scratch, n_steps=400)

In [ ]:
def smooth(a, w=20):
    return np.convolve(a, np.ones(w)/w, 'valid')

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Pre-training loss
axes[0].plot(smooth(pretrain_losses), 'steelblue', lw=2)
axes[0].axhline(np.log(VOCAB_SIZE), color='tomato', linestyle=':', lw=1.5, label='random chance')
axes[0].set_xlabel('Pre-training step'); axes[0].set_ylabel('MLM loss')
axes[0].set_title('BERT MLM pre-training\n(predict masked tokens)')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

# Fine-tuning loss comparison
axes[1].plot(smooth(ft_losses,      20), 'steelblue', lw=2, label='Pre-trained BERT')
axes[1].plot(smooth(scratch_losses, 20), 'tomato',    lw=2, label='From scratch')
axes[1].set_xlabel('Fine-tuning step'); axes[1].set_ylabel('Classification loss')
axes[1].set_title('Fine-tuning loss\n(same labeled data)')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

# Validation accuracy
if ft_accs and scratch_accs:
    steps_ft,     accs_ft     = zip(*ft_accs)
    steps_scratch, accs_scratch = zip(*scratch_accs)
    axes[2].plot(steps_ft,      accs_ft,      'steelblue', lw=2, marker='o', label='Pre-trained BERT')
    axes[2].plot(steps_scratch, accs_scratch, 'tomato',    lw=2, marker='s', label='From scratch')
    axes[2].axhline(0.5, color='k', linestyle=':', lw=1, label='random chance')
    axes[2].set_xlabel('Fine-tuning step'); axes[2].set_ylabel('Validation accuracy')
    axes[2].set_title('Pre-training helps: faster convergence\nand higher accuracy')
    axes[2].set_ylim(0, 1.05); axes[2].legend(); axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('10_bert_results.png', bbox_inches='tight')
plt.show()

In [ ]:
# Visualize bidirectional attention — the key difference from GPT
# In GPT, position 0 can only see position 0. In BERT, it sees all positions.

bert.eval()
example = "To be or not to be"
tok_ids = [CLS_ID] + [char2id.get(c, N_SPECIAL) for c in example] + [SEP_ID]
if len(tok_ids) < MAX_LEN:
    tok_ids += [PAD_ID] * (MAX_LEN - len(tok_ids))
x_ex = torch.tensor([tok_ids[:MAX_LEN]], dtype=torch.long).to(device)
T_ex = min(len(example) + 2, MAX_LEN)  # CLS + text + SEP
labels_ex = ['[CLS]'] + list(example) + ['[SEP]']

# Capture attention weights from first BERT block
attn_weights = None
hook_handle = None

def capture_attn(module, input, output):
    global attn_weights
    # Recompute weights from the stored Q, K in the attention module
    pass  # We'll compute separately below

with torch.no_grad():
    # Manually extract first-block attention weights
    pad_mask = (x_ex == PAD_ID)
    pos = torch.arange(x_ex.size(1), device=device).unsqueeze(0)
    h = bert.tok_emb(x_ex) + bert.pos_emb(pos)
    h = bert.drop(h)
    h = bert.ln_f(h)  # skip for now, compute raw attention

    # Re-derive through first block for attention maps
    first_block = bert.blocks[0]
    h_ln = first_block.ln1(bert.drop(bert.tok_emb(x_ex) + bert.pos_emb(pos)))
    B, T, D = h_ln.shape
    H = N_HEADS; dk = D_MODEL // N_HEADS
    Wqkv = first_block.attn.Wqkv
    Q, K, V = Wqkv(h_ln).split(D_MODEL, dim=-1)
    Q = Q.view(B, T, H, dk).transpose(1, 2)
    K = K.view(B, T, H, dk).transpose(1, 2)
    scores = Q @ K.transpose(-2, -1) / math.sqrt(dk)
    scores = scores.masked_fill(pad_mask.unsqueeze(1).unsqueeze(2), float('-inf'))
    attn_weights = F.softmax(scores, dim=-1)  # (1, H, T, T)

fig, axes = plt.subplots(1, N_HEADS, figsize=(16, 3))
for h_idx in range(N_HEADS):
    ax = axes[h_idx]
    w = attn_weights[0, h_idx, :T_ex, :T_ex].cpu().numpy()
    ax.imshow(w, cmap='Blues', vmin=0, vmax=w.max())
    ax.set_title(f'Head {h_idx+1}', fontsize=9)
    ax.set_xticks(range(T_ex)); ax.set_xticklabels(labels_ex[:T_ex], fontsize=6, rotation=70)
    ax.set_yticks(range(T_ex)); ax.set_yticklabels(labels_ex[:T_ex], fontsize=6)

plt.suptitle('BERT bidirectional attention — each position sees ALL others (including future)',
             fontsize=11)
plt.tight_layout()
plt.savefig('10_bert_attention.png', bbox_inches='tight')
plt.show()

## GPT vs. BERT: The Core Architectural Difference

| Property | GPT (decoder-only) | BERT (encoder-only) |
|---|---|---|
| **Attention mask** | Causal (left-to-right only) | Bidirectional (all to all) |
| **Pre-training** | Causal language modeling | Masked language modeling |
| **Strengths** | Text generation, open-ended completion | Classification, extraction, NLU |
| **[CLS] token** | Not used | Summary embedding for classification |
| **Parallelism** | Full (during training) | Full |
| **Context** | Only past tokens | Both past and future tokens |
| **Scale** | GPT-3: 175B params | BERT-large: 340M params |

**Why GPT won the long game**: the text generation paradigm is more general.
By training at much larger scale (GPT-3, GPT-4, Claude), the decoder-only
architecture can perform understanding tasks via in-context learning — you
don't need fine-tuning. BERT still dominates for embedding-heavy applications
and tasks where you explicitly have labeled data.

## Key Takeaways

| Concept | What we learned |
|---|---|
| **MLM objective** | Mask 15% of tokens; predict from both directions — richer representations |
| **Bidirectional attention** | No causal mask — every position sees every other |
| **[CLS] token** | Acts as sentence summary; fine-tuned for classification |
| **80/10/10 masking** | Prevents model from relying on `[MASK]` signal alone |
| **Weight tying** | Output projection = embedding matrix — efficient and effective |
| **Pre-training transfers** | Faster fine-tuning convergence + higher accuracy vs. random init |
| **Fine-tuning** | Add a small head; update all weights on labeled data |

---

## The Complete Journey

| Notebook | Architecture | Key Innovation |
|---|---|---|
| 01 Perceptron | Single neuron | Linear decision boundary, learning rule |
| 02 MLP | Stacked layers | Backprop, universal approximation, XOR |
| 03 CNN | Convolutional + pooling | Weight sharing over space, feature hierarchy |
| 04 RNN | Recurrent hidden state | Weight sharing over time, BPTT |
| 05 LSTM | Gated cell state | Additive updates prevent vanishing gradient |
| 06 Seq2seq | Encoder + decoder | Variable-length output; fixed-context bottleneck |
| 07 Attention | Bahdanau attention | Per-step context; eliminates bottleneck |
| 08 Transformer | Self-attention + FFN | Fully parallel; O(1) path between any two tokens |
| 09 GPT | Causal decoder | Autoregressive LM; scales to 175B params |
| 10 BERT | Bidirectional encoder | MLM pre-training; fine-tune on labeled tasks |